# Mini Implementation: Cybersecurity & Privacy in AI-Enabled Agricultural IoT
A small, beginner-friendly demo of the ideas in *"Cybersecurity and Privacy in AI-Enabled Agricultural IoT Ecosystems: A Systematic Review"* (Algorithms 2026, 19, 827).

It covers four parts, all on toy data so it runs in seconds:
1. Check the PRISMA study-selection arithmetic (2535 records to 103 studies)
2. Plot the paper's reported threat, privacy, and solution coverage
3. Retrospective **STRIDE** keyword coding plus inter-rater reliability (Cohen's kappa)
4. A rule-based **ecological validity** rater (High / Moderate / Low)

Note: the data in parts 3 and 4 is synthetic, written for this demo. It is not the paper's dataset.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import cohen_kappa_score
rng = np.random.default_rng(42)

## 1. PRISMA flow (numbers from Table 7)

In [ ]:
steps = [("Retrieved", 2535), ("After dedup (-520)", 2535-520),
         ("After title/abstract (-1195)", 2015-1195),
         ("After 1st full-text (-117)", 820-117),
         ("After 2nd full-text (-598)", 703-598),
         ("After citation tracking (+8, -3, -7)", 105+8-3-7)]
for name, n in steps: print(f"{name:40s} {n}")
assert steps[-1][1] == 103, "Should end at 103 included studies"

plt.figure(figsize=(8,3.5))
plt.barh([s[0] for s in steps][::-1], [s[1] for s in steps][::-1], color="seagreen")
plt.xlabel("Records"); plt.title("PRISMA funnel: 2535 records to 103 studies")
plt.tight_layout(); plt.show()

## 2. Reported coverage across the 103 studies (from Sections 4.2 to 4.4)

In [ ]:
threats = {"DoS/DDoS":69.9,"Tampering":52.4,"Spoofing":45.6,"MitM":45.6,"Eavesdropping":38.8,
           "Malware":35.9,"Ransomware":27.2,"Replay":26.2,"Jamming":25.2}
privacy = {"Disclosure of Info":68.0,"Non-Compliance":32.0,"Linkability":12.6,"Unawareness":6.8,"Detectability":1.0}
solutions = {"Crypto/Auth":62.1,"AI/ML detection":44.7,"Blockchain":42.7,"Broader AI/ML":27.2,
             "Governance":20.4,"Federated learning":16.5,"Differential privacy":4.9}
layers = {"Cloud":78.6,"Device":59.2,"Network":57.3,"Edge/Fog":27.2}

fig, ax = plt.subplots(2, 2, figsize=(12,7))
for a,(t,d,c) in zip(ax.ravel(), [("Threats (RQ1)",threats,"firebrick"),("Privacy / LINDDUN (RQ2)",privacy,"darkorange"),
                                  ("Safeguards (RQ3)",solutions,"steelblue"),("Architecture layers",layers,"purple")]):
    a.barh(list(d)[::-1], list(d.values())[::-1], color=c); a.set_title(t); a.set_xlabel("% of studies")
plt.tight_layout(); plt.show()
print("Key gap: edge layer is only", layers["Edge/Fog"], "% of studies; only 7.8% have High ecological validity.")

## 3. Retrospective STRIDE coding + reliability
The paper maps each study to STRIDE categories using an equivalence dictionary, then two reviewers code 20 studies and agreement is measured per category (multi-label, so binary present/absent kappa).

Here: a small keyword dictionary, 20 synthetic abstracts, and a second "reviewer" simulated by randomly flipping about 10% of labels.

In [ ]:
stride_dict = {
 "Spoofing": ["impersonat", "forged", "masquerad", "spoof"],
 "Tampering": ["falsif", "tamper", "modif", "inject"],
 "Repudiation": ["audit", "deny", "accountab"],
 "Information Disclosure": ["eavesdrop", "intercept", "leak", "expos"],
 "Denial of Service": ["ddos", "flood", "jamming", "denial-of-service"],
 "Elevation of Privilege": ["privilege escalation", "unauthorized control", "root access"],
}
snippets = {
 "Spoofing": "An attacker impersonates a legitimate soil sensor to the gateway.",
 "Tampering": "Falsified moisture readings trigger wrong irrigation decisions.",
 "Repudiation": "Weak audit logs make it hard to know who issued a command.",
 "Information Disclosure": "Eavesdropping on LoRaWAN traffic leaks farm yield data.",
 "Denial of Service": "A DDoS flood takes the irrigation controller offline.",
 "Elevation of Privilege": "Privilege escalation gives a compromised node control of actuators.",
}
cats = list(stride_dict)

# build 20 toy abstracts with known ground-truth labels
true = np.zeros((20, len(cats)), int); texts = []
for i in range(20):
    pick = rng.choice(len(cats), size=rng.integers(1, 4), replace=False)
    true[i, pick] = 1
    texts.append(" ".join(snippets[cats[j]] for j in pick))

def code_text(t):
    t = t.lower()
    return [int(any(k in t for k in stride_dict[c])) for c in cats]

A = np.array([code_text(t) for t in texts])            # reviewer 1: keyword dictionary
flip = rng.random(A.shape) < 0.10
B = np.where(flip, 1 - A, A)                           # reviewer 2: noisy human-like coder

rows = []
for j, c in enumerate(cats):
    agree = (A[:, j] == B[:, j]).mean() * 100
    kappa = cohen_kappa_score(A[:, j], B[:, j]) if len(set(A[:, j])|set(B[:, j])) > 1 else np.nan
    rows.append((c, f"{agree:.0f}%", round(kappa, 3) if kappa == kappa else "not estimable"))
print(pd.DataFrame(rows, columns=["STRIDE category", "Observed agreement", "Cohen's kappa"]).to_string(index=False))
print("\nAs in the paper, report observed agreement next to kappa, since kappa breaks down when a category is near-constant.")

## 4. Ecological validity rater
Simplified version of the paper's rule (Section 3.6):
- **High**: real agricultural data or hardware in a genuine field, at least 2 agricultural constraints, and results reported under those conditions
- **Moderate**: agricultural data or hardware in a lab, simulation or testbed
- **Low**: pure simulation, generic non-agricultural data, or purely analytical work

In [ ]:
def eco_validity(env, ag_data, n_constraints, reported_under_conditions):
    if env == "field" and ag_data and n_constraints >= 2 and reported_under_conditions:
        return "High"
    if ag_data and env in ("lab", "testbed", "field"):   # field but missing criteria stays conservative
        return "Moderate"
    return "Low"

studies = pd.DataFrame([
 ("Live-attack threat intel on real smart farm", "field", True, 3, True),
 ("Federated IDS on generic network dataset",   "simulation", False, 0, False),
 ("Lightweight auth on lab Raspberry Pi nodes", "lab", True, 1, True),
 ("Blockchain traceability, math proof only",   "analytical", False, 0, False),
 ("LoRaWAN replay attack on irrigation testbed","testbed", True, 1, True),
 ("CNN IDS, 98% on benchmark dataset",          "simulation", False, 0, False),
 ("Field trial of sensor auth, partial report", "field", True, 2, False),
 ("Drone spoofing test in open field",          "field", True, 2, True),
], columns=["study", "env", "ag_data", "n_constraints", "reported"])

studies["ecological_validity"] = [eco_validity(r.env, r.ag_data, r.n_constraints, r.reported) for r in studies.itertuples()]
print(studies[["study", "ecological_validity"]].to_string(index=False))

counts = studies["ecological_validity"].value_counts().reindex(["High","Moderate","Low"]).fillna(0)
counts.plot(kind="bar", color=["green","gold","tomato"], rot=0, title="Toy ecological-validity distribution")
plt.ylabel("Studies"); plt.show()

## Takeaways
- Selection arithmetic checks out: 2535 to 103 studies.
- Threats cluster around DoS, tampering and spoofing; the **edge layer** and **adversarial attacks on agricultural AI** are under-studied.
- Privacy work leans on confidentiality, with 90.3% of studies citing no regulation.
- Only **7.8%** of studies have High ecological validity, so lab accuracy (for example 98% IDS) does not guarantee real-farm performance.

**Ideas to extend:** train a small anomaly detector (Isolation Forest) on simulated soil-sensor data, then inject tampered readings to see detection; or add a small adversarial perturbation to a crop-disease classifier. Both connect to the paper's AI dual-role argument.